# FusedHexapodModel V5.0 — Deploy Export

**Pipeline:**
1. SWA über die letzten 5 Checkpoints (gleichgewichtetes Averaging)
2. RepConv-Folding (`switch_to_deploy()` auf alle RepConv-Layer)
3. Deploy-Mode für alle Module aktivieren
4. Deploy-Checkpoint speichern
5. 3-HEF ONNX-Split:
   - **Backbone HEF** — MNV2-1.4 + ChannelAligner → `f_s4, f_s8, f_s16, f_s32`
   - **Geometry HEF** — GeoStem + NormalsHead + CorrelationStereoHead → `disp_s4, normals_s4, disp_s8`
   - **Detection HEF** — SPPF + FPN + LRASPPHead + YOLOHead → `seg, yolo_s8, yolo_s16, yolo_s32`
6. onnxsim für alle 3 Dateien
7. Numerische Validierung (PyTorch vs ONNX Runtime)

## Zelle 1 — Imports & Device

In [32]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import numpy as np
import os, glob, re, copy, subprocess, math

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

Device: cuda
  GPU: NVIDIA GeForce RTX 3080 Ti
  VRAM: 12.0 GB


## Zelle 2 — CONFIG (exakt wie Trainingsscript)

In [33]:
# =====================================================================
# V5 CONFIG — muss exakt dem Trainingsscript entsprechen!
# =====================================================================
CONFIG = {
    'img_height': 480, 'img_width': 640,
    'num_det_classes': 41,   # 40 robot-relevant classes + 1 "unknown object"
    'num_seg_classes': 6,
    'max_disp_pixel': 192,
    'backbone_stride': 4,
    'internal_disp_steps': 24,
    'tartan_fx': 320.0,
    'tartan_fy': 320.0,
    'tartan_baseline': 0.25,
    'seg_class_weights': [1.0, 2.0, 1.0, 1.0, 0.5, 1.0],
    'VIS_THRESH': 0.30,
    'save_dir': './checkpoints',
    # Deploy-Flag — wird im SWA-Block zunächst auf False belassen!
    'deploy': False,
}

V5_CONFIG = {
    'use_correlation_stereo': True,
    'channel_alignment': 'stereo_focused',  # 64/128/128/256
}

CONFIG['v5'] = V5_CONFIG

SEG_CLASS_NAMES = ['WALKABLE', 'STEP', 'WALL', 'OBSTACLE', 'VOID', 'TERRAIN']

ROBOT_CAT_IDS = [1,2,3,4,6,8,10,11,13,14,15,16,17,18,27,28,31,33,44,47,51,
                 62,63,64,65,67,70,72,73,75,76,77,78,79,81,82,84,85,86,88]
UNKNOWN_CLASS_ID = 40

print(f'V5.0 Config: {CONFIG["num_det_classes"]} det classes, {CONFIG["num_seg_classes"]} seg classes')
print(f'Channel alignment: {V5_CONFIG["channel_alignment"]}')
print(f'Stereo head: {"CorrelationStereoHead" if V5_CONFIG["use_correlation_stereo"] else "HierarchicalStereoHead"}')

V5.0 Config: 41 det classes, 6 seg classes
Channel alignment: stereo_focused
Stereo head: CorrelationStereoHead


## Zelle 3 — Modell-Architektur (V5.0)

Alle Klassen exakt aus `FusedHexapodModel_V5_0-Phase-1.py`.

In [34]:
# ─────────────────────────────────────────────────────────────────────
# BUILDING BLOCKS
# ─────────────────────────────────────────────────────────────────────
class LearnablePool(nn.Module):
    def __init__(self, ch, kernel_size, stride=None):
        super().__init__()
        stride = stride or kernel_size
        self.pool = nn.Conv2d(ch, ch, kernel_size, stride=stride, groups=ch, bias=False)
        with torch.no_grad():
            k = kernel_size if isinstance(kernel_size, int) else kernel_size[0]*kernel_size[1]
            self.pool.weight.fill_(1.0 / (k if isinstance(kernel_size, int) else k))
    def forward(self, x): return self.pool(x)

class DWSepConv(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, stride=1, padding=0, bias=True, dilation=1):
        super().__init__()
        self.dw = nn.Conv2d(in_ch, in_ch, kernel_size, stride=stride, padding=padding,
                            dilation=dilation, groups=in_ch, bias=False)
        self.pw = nn.Conv2d(in_ch, out_ch, 1, bias=bias)
    def forward(self, x): return self.pw(self.dw(x))

FPN_CH = 64

class AddCoords(nn.Module):
    def __init__(self, h, w, deploy=False):
        super().__init__()
        self.deploy = deploy
        y = torch.linspace(-1, 1, h).view(1,1,h,1).expand(1,1,h,w).contiguous().clone()
        x = torch.linspace(-1, 1, w).view(1,1,1,w).expand(1,1,h,w).contiguous().clone()
        self.register_buffer('y_coords', y)
        self.register_buffer('x_coords', x)
    def forward(self, x_in):
        if self.deploy:
            return torch.cat([x_in, self.y_coords, self.x_coords], dim=1)
        b = x_in.shape[0]
        return torch.cat([x_in,
                          self.y_coords.expand(b,-1,-1,-1),
                          self.x_coords.expand(b,-1,-1,-1)], dim=1)

class CoordConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, h, w, deploy=False,
                 kernel_size=3, stride=1, padding=1, bias=False):
        super().__init__()
        self.add_coords = AddCoords(h, w, deploy=deploy)
        self.conv = nn.Conv2d(in_channels+2, out_channels, kernel_size,
                              stride=stride, padding=padding, bias=bias)
    def forward(self, x): return self.conv(self.add_coords(x))

class RepConv(nn.Module):
    def __init__(self, c1, c2, kernel_size=3, stride=1, padding=1, deploy=False):
        super().__init__()
        self.deploy = deploy
        self.c1, self.c2 = c1, c2
        self.stride, self.padding = stride, padding
        self.act = nn.ReLU(inplace=True)
        if deploy:
            self.rbr_reparam = nn.Conv2d(c1, c2, kernel_size, stride, padding, bias=True)
        else:
            self.rbr_identity = nn.BatchNorm2d(c1) if c2==c1 and stride==1 else None
            self.rbr_dense = nn.Sequential(
                nn.Conv2d(c1, c2, kernel_size, stride, padding, bias=False), nn.BatchNorm2d(c2))
            self.rbr_1x1 = nn.Sequential(
                nn.Conv2d(c1, c2, 1, stride, 0, bias=False), nn.BatchNorm2d(c2))

    def forward(self, x):
        if self.deploy:
            return self.act(self.rbr_reparam(x))
        id_out = 0 if self.rbr_identity is None else self.rbr_identity(x)
        return self.act(self.rbr_dense(x) + self.rbr_1x1(x) + id_out)

    def get_equivalent_kernel_bias(self):
        k3, b3 = self._fuse_bn_tensor(self.rbr_dense)
        k1, b1 = self._fuse_bn_tensor(self.rbr_1x1)
        k1 = F.pad(k1, [1,1,1,1])
        ki, bi = self._fuse_bn_tensor(self.rbr_identity)
        return k3+k1+ki, b3+b1+bi

    def _fuse_bn_tensor(self, branch):
        dev = self.rbr_dense[0].weight.device
        if branch is None:
            return (torch.zeros((self.c2, self.c1, 3, 3), device=dev),
                    torch.zeros(self.c2, device=dev))
        if isinstance(branch, nn.BatchNorm2d):
            kernel = torch.zeros((self.c1, self.c1, 3, 3), device=branch.weight.device)
            for i in range(self.c1): kernel[i, i, 1, 1] = 1.0
            return self._fuse_bn(kernel, branch.running_mean, branch.running_var,
                                 branch.weight, branch.bias, branch.eps)
        return self._fuse_bn(branch[0].weight, branch[1].running_mean, branch[1].running_var,
                             branch[1].weight, branch[1].bias, branch[1].eps)

    def _fuse_bn(self, kernel, mean, var, gamma, beta, eps):
        std = (var + eps).sqrt()
        t = (gamma / std).reshape(-1, 1, 1, 1)
        return kernel * t, beta - mean * gamma / std

    def switch_to_deploy(self):
        if self.deploy: return
        kernel, bias = self.get_equivalent_kernel_bias()
        self.rbr_reparam = nn.Conv2d(self.c1, self.c2, 3, self.stride, self.padding, bias=True)
        self.rbr_reparam.weight.data = kernel
        self.rbr_reparam.bias.data   = bias
        for attr in ['rbr_dense', 'rbr_1x1', 'rbr_identity']:
            if hasattr(self, attr): delattr(self, attr)
        self.deploy = True


# ─────────────────────────────────────────────────────────────────────
# FPN NECK + SPPF
# ─────────────────────────────────────────────────────────────────────
class LightFPNNeck(nn.Module):
    def __init__(self, ch_s8, ch_s16, ch_s32, fpn_ch=FPN_CH):
        super().__init__()
        self.lat_s8  = nn.Conv2d(ch_s8,  fpn_ch, 1, bias=False)
        self.lat_s16 = nn.Conv2d(ch_s16, fpn_ch, 1, bias=False)
        self.lat_s32 = nn.Conv2d(ch_s32, fpn_ch, 1, bias=False)
        self.smooth_s8  = RepConv(fpn_ch, fpn_ch)
        self.smooth_s16 = RepConv(fpn_ch, fpn_ch)
        self.bu_s16 = RepConv(fpn_ch, fpn_ch, stride=2)
        self.bu_s32 = RepConv(fpn_ch, fpn_ch, stride=2)

    def forward(self, f_s8, f_s16, f_s32):
        p32 = self.lat_s32(f_s32)
        p16 = self.lat_s16(f_s16) + F.interpolate(p32, scale_factor=2, mode='nearest')
        p8  = self.lat_s8(f_s8)   + F.interpolate(p16, scale_factor=2, mode='nearest')
        p8  = self.smooth_s8(p8)
        p16 = self.smooth_s16(p16) + self.bu_s16(p8)
        p32 = p32 + self.bu_s32(p16)
        return p8, p16, p32

class SPPF(nn.Module):
    def __init__(self, c1, c2, k=5):
        super().__init__()
        c_ = c1 // 2
        self.cv1 = nn.Sequential(nn.Conv2d(c1, c_, 1, 1, bias=False), nn.BatchNorm2d(c_), nn.ReLU6(inplace=True))
        self.cv2 = nn.Sequential(nn.Conv2d(c_*4, c2, 1, 1, bias=False), nn.BatchNorm2d(c2), nn.ReLU6(inplace=True))
        self.m = nn.MaxPool2d(kernel_size=k, stride=1, padding=k//2)
    def forward(self, x):
        x = self.cv1(x)
        y1, y2, y3 = self.m(x), self.m(self.m(x)), self.m(self.m(self.m(x)))
        return self.cv2(torch.cat((x, y1, y2, y3), 1))

class GeometryStem(nn.Module):
    def __init__(self, in_ch=32, out_ch=32):
        super().__init__()
        self.stem = nn.Sequential(
            RepConv(in_ch, out_ch, kernel_size=3, stride=1, padding=1), nn.ReLU(inplace=False),
            RepConv(out_ch, out_ch, kernel_size=3, stride=1, padding=1), nn.ReLU(inplace=False))
    def forward(self, x): return self.stem(x)


# ─────────────────────────────────────────────────────────────────────
# STEREO HEADS
# ─────────────────────────────────────────────────────────────────────
class RefinementStage(nn.Module):
    def __init__(self, guidance_channels, scale_factor, use_edge_guidance=False, deploy=False):
        super().__init__()
        self.deploy = deploy
        self.scale_factor = scale_factor
        self.use_edge_guidance = use_edge_guidance
        extra = 1 if use_edge_guidance else 0
        self.net = nn.Sequential(
            nn.Conv2d(1 + guidance_channels + extra, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 3, padding=1))
        kx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32).view(1,1,3,3)
        ky = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32).view(1,1,3,3)
        self.register_buffer('kx', kx)
        self.register_buffer('ky', ky)

    def _edge_map(self, img):
        return torch.abs(F.conv2d(img, self.kx, padding=1)) + torch.abs(F.conv2d(img, self.ky, padding=1))

    def forward(self, disparity_low, guidance, max_disp, gray_img=None):
        mode = 'nearest' if self.deploy else 'bilinear'
        disparity_up = F.interpolate(
            disparity_low, scale_factor=self.scale_factor, mode=mode,
            align_corners=False if mode == 'bilinear' else None) * self.scale_factor
        norm_disp = disparity_up / max_disp
        inp = [norm_disp, guidance]
        if self.use_edge_guidance:
            assert gray_img is not None
            if gray_img.shape[-2:] != disparity_up.shape[-2:]:
                gray_img = F.interpolate(gray_img, size=disparity_up.shape[-2:],
                                         mode='nearest' if self.deploy else 'bilinear',
                                         align_corners=None if self.deploy else False)
            inp.append(self._edge_map(gray_img))
        return F.relu(disparity_up + self.net(torch.cat(inp, dim=1)))


class CoarseCostVolume(nn.Module):
    """V4.0 CostVolume — bleibt für Checkpoint-Kompatibilität erhalten."""
    def __init__(self, max_disp, in_channels, deploy=False):
        super().__init__()
        self.max_disp = max_disp
        self.deploy   = deploy
        self.corr = nn.Conv2d(in_channels * 2, 1, 1, bias=True)
        if self.deploy:
            self.corr_grouped = nn.Conv2d(max_disp * in_channels * 2, max_disp, 1, groups=max_disp)

    def forward(self, feat_l, feat_r):
        if self.deploy:
            all_shifted = []
            for d in range(self.max_disp):
                shifted = feat_r if d == 0 else F.pad(feat_r, (d,0,0,0))[:,:,:,:-d]
                all_shifted.append(torch.cat([feat_l, shifted], dim=1))
            return self.corr_grouped(torch.cat(all_shifted, dim=1))
        B, C, H, W = feat_l.shape
        cost_slices = []
        for d in range(self.max_disp):
            if d == 0:
                cost_slices.append(torch.cat([feat_l, feat_r], dim=1))
            else:
                shifted = torch.zeros_like(feat_r)
                shifted[:,:,:,d:] = feat_r[:,:,:,:-d]
                cost_slices.append(torch.cat([feat_l, shifted], dim=1))
        cost = torch.stack(cost_slices, dim=2)
        B, C2, D, H, W = cost.shape
        cost = cost.permute(0,2,1,3,4).reshape(B*D, C2, H, W)
        return self.corr(cost).view(B, D, H, W)


class HailoCostVolume(nn.Module):
    """V5.0 CostVolume: Shift & Concat — kein pixelweises Dot-Product."""
    def __init__(self, max_disp, deploy=False):
        super().__init__()
        self.max_disp = max_disp
        self.deploy   = deploy

    def forward(self, feat_l, feat_r):
        B, C, H, W = feat_l.shape
        if self.deploy:
            slices = []
            for d in range(self.max_disp):
                shifted = feat_r if d == 0 else F.pad(feat_r, (d,0,0,0))[:,:,:,:-d]
                slices.append(torch.cat([feat_l, shifted], dim=1))
            return torch.cat(slices, dim=1)
        volume = torch.zeros(B, self.max_disp*2*C, H, W, device=feat_l.device, dtype=feat_l.dtype)
        for d in range(self.max_disp):
            s, e = d*2*C, d*2*C + 2*C
            if d == 0:
                volume[:,s:e] = torch.cat([feat_l, feat_r], dim=1)
            else:
                shifted = torch.zeros_like(feat_r)
                shifted[:,:,:,d:] = feat_r[:,:,:,:-d]
                volume[:,s:e] = torch.cat([feat_l, shifted], dim=1)
        return volume


class HierarchicalStereoHead(nn.Module):
    """V4.0 Stereo Head — für Checkpoint-Kompatibilität."""
    def __init__(self, ch_s8, ch_s4, max_disp_s8, use_normals=True, deploy=False):
        super().__init__()
        self.max_disp_s8 = max_disp_s8
        self.use_normals = use_normals
        self.deploy      = deploy
        self.reduce_s8   = CoordConv2d(ch_s8+32, 16, h=60, w=80, deploy=deploy, kernel_size=1, padding=0, bias=False)
        s4_guidance_ch   = ch_s4+32+(3 if use_normals else 0)
        self.reduce_s4   = CoordConv2d(s4_guidance_ch, 16, h=120, w=160, deploy=deploy, kernel_size=1, padding=0, bias=False)
        self.stereo_coarse = CoarseCostVolume(max_disp=max_disp_s8, in_channels=16, deploy=deploy)
        self.stereo_refine_s4 = RefinementStage(guidance_channels=16, scale_factor=2.0, deploy=deploy)
        self.register_buffer('disp_reg', torch.arange(max_disp_s8, dtype=torch.float32).view(1,-1,1,1))
        self.temperature    = 0.7
        self.context_weight = 0.8
        self.context = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(inplace=True),
            DWSepConv(16, 16, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, 3, padding=1))
        if self.deploy:
            self.context_grouped = nn.Sequential(
                nn.Conv2d(max_disp_s8, max_disp_s8*16, 3, padding=1, groups=max_disp_s8), nn.ReLU(inplace=True),
                nn.Conv2d(max_disp_s8*16, max_disp_s8*16, 3, padding=1, groups=max_disp_s8*16, bias=False),
                nn.Conv2d(max_disp_s8*16, max_disp_s8*16, 1, groups=max_disp_s8), nn.ReLU(inplace=True),
                nn.Conv2d(max_disp_s8*16, max_disp_s8, 3, padding=1, groups=max_disp_s8))
        self.geo_downsample = nn.Conv2d(32, 32, kernel_size=3, stride=2, padding=1, bias=False)

    def forward(self, l_s8, r_s8, l_s4, l_img_raw, normals_s4=None, geo_features_l=None, geo_features_r=None):
        if geo_features_l is None:
            geo_features_l = torch.zeros(l_s4.size(0), 32, l_s4.size(2), l_s4.size(3), device=l_s4.device, dtype=l_s4.dtype)
        if geo_features_r is None:
            geo_features_r = torch.zeros_like(geo_features_l)
        geo_l_s8 = self.geo_downsample(geo_features_l)
        geo_r_s8 = self.geo_downsample(geo_features_r)
        feat_l_s8 = self.reduce_s8(torch.cat([l_s8, geo_l_s8], dim=1))
        feat_r_s8 = self.reduce_s8(torch.cat([r_s8, geo_r_s8], dim=1))
        norm_in = normals_s4 if (self.use_normals and normals_s4 is not None) else \
            torch.zeros(l_s4.size(0), 3, l_s4.size(2), l_s4.size(3), device=l_s4.device, dtype=l_s4.dtype)
        l_s4_combined = torch.cat([l_s4, geo_features_l, norm_in], dim=1) if self.use_normals else \
            torch.cat([l_s4, geo_features_l], dim=1)
        feat_l_s4 = self.reduce_s4(l_s4_combined)
        vol_s8 = self.stereo_coarse(feat_l_s8, feat_r_s8)
        if self.deploy:
            vol_ctx = self.context_grouped(vol_s8)
        else:
            B, D, H, W = vol_s8.shape
            vol_ctx = self.context(vol_s8.view(B*D, 1, H, W)).view(B, D, H, W)
        vol_s8 = self.context_weight * vol_ctx + (1.0 - self.context_weight) * vol_s8
        if self.deploy:
            vol_scaled = vol_s8 / self.temperature
            v_max, _ = torch.max(vol_scaled, dim=1, keepdim=True)
            v_exp = torch.exp(vol_scaled - v_max)
            prob_s8 = v_exp / (torch.sum(v_exp, dim=1, keepdim=True) + 1e-6)
        else:
            prob_s8 = F.softmax(vol_s8 / self.temperature, dim=1)
        disp_s8 = torch.sum(prob_s8 * self.disp_reg, dim=1, keepdim=True)
        if not self.training:
            confidence = prob_s8.max(dim=1, keepdim=True)[0]
            disp_s8 = disp_s8 * torch.clamp((confidence - 0.10) / 0.10, 0.0, 1.0)
        disp_s4 = self.stereo_refine_s4(disp_s8, feat_l_s4, max_disp=self.max_disp_s8 * 2.0)
        return disp_s4, disp_s8


class CorrelationStereoHead(nn.Module):
    """V5.0 Stereo Head: HailoCostVolume + 2-stufige 1x1-Reduktion."""
    def __init__(self, ch_s8, ch_s4, max_disp_s8, use_normals=True, deploy=False):
        super().__init__()
        self.max_disp_s8 = max_disp_s8
        self.use_normals = use_normals
        self.deploy      = deploy
        self.reduce_s8   = CoordConv2d(ch_s8+32, 16, h=60, w=80, deploy=deploy, kernel_size=1, padding=0, bias=False)
        s4_guidance_ch   = ch_s4+32+(3 if use_normals else 0)
        self.reduce_s4   = CoordConv2d(s4_guidance_ch, 16, h=120, w=160, deploy=deploy, kernel_size=1, padding=0, bias=False)
        self.cost_volume  = HailoCostVolume(max_disp=max_disp_s8, deploy=deploy)
        self.cv_reduction = nn.Sequential(
            nn.Conv2d(32*max_disp_s8, 4*max_disp_s8, 1, bias=False), nn.ReLU6(inplace=True),
            nn.Conv2d(4*max_disp_s8, max_disp_s8, 1, bias=False))
        self.post_corr = nn.Sequential(
            nn.Conv2d(max_disp_s8, max_disp_s8, 3, padding=1), nn.ReLU6(inplace=True),
            nn.Conv2d(max_disp_s8, max_disp_s8, 3, padding=1), nn.ReLU6(inplace=True))
        self.context_weight   = 0.8
        self.stereo_refine_s4 = RefinementStage(guidance_channels=16, scale_factor=2.0, deploy=deploy)
        self.register_buffer('disp_reg', torch.arange(max_disp_s8, dtype=torch.float32).view(1,-1,1,1))
        self.temperature    = 0.5
        self.geo_downsample = nn.Conv2d(32, 32, kernel_size=3, stride=2, padding=1, bias=False)

    def forward(self, l_s8, r_s8, l_s4, l_img_raw, normals_s4=None, geo_features_l=None, geo_features_r=None):
        if geo_features_l is None:
            geo_features_l = torch.zeros(l_s4.size(0), 32, l_s4.size(2), l_s4.size(3), device=l_s4.device, dtype=l_s4.dtype)
        if geo_features_r is None:
            geo_features_r = torch.zeros_like(geo_features_l)
        geo_l_s8 = self.geo_downsample(geo_features_l)
        geo_r_s8 = self.geo_downsample(geo_features_r)
        feat_l_s8 = self.reduce_s8(torch.cat([l_s8, geo_l_s8], dim=1))
        feat_r_s8 = self.reduce_s8(torch.cat([r_s8, geo_r_s8], dim=1))
        norm_in = normals_s4 if (self.use_normals and normals_s4 is not None) else \
            torch.zeros(l_s4.size(0), 3, l_s4.size(2), l_s4.size(3), device=l_s4.device, dtype=l_s4.dtype)
        l_s4_combined = torch.cat([l_s4, geo_features_l, norm_in], dim=1) if self.use_normals else \
            torch.cat([l_s4, geo_features_l], dim=1)
        feat_l_s4 = self.reduce_s4(l_s4_combined)
        feat_l_norm = F.normalize(feat_l_s8, p=2, dim=1, eps=1e-4)
        feat_r_norm = F.normalize(feat_r_s8, p=2, dim=1, eps=1e-4)
        raw_volume  = self.cost_volume(feat_l_norm, feat_r_norm)
        corr        = self.cv_reduction(raw_volume)
        corr_refined = self.post_corr(corr)
        vol_s8 = self.context_weight * corr_refined + (1.0 - self.context_weight) * corr
        if self.deploy:
            vol_scaled = vol_s8 / self.temperature
            v_max, _ = torch.max(vol_scaled, dim=1, keepdim=True)
            v_exp = torch.exp(vol_scaled - v_max)
            prob_s8 = v_exp / (torch.sum(v_exp, dim=1, keepdim=True) + 1e-6)
        else:
            prob_s8 = F.softmax(vol_s8 / self.temperature, dim=1)
        disp_s8 = torch.sum(prob_s8 * self.disp_reg, dim=1, keepdim=True)
        if not self.training:
            confidence = prob_s8.max(dim=1, keepdim=True)[0]
            disp_s8 = disp_s8 * torch.clamp((confidence - 0.10) / 0.10, 0.0, 1.0)
        disp_s4 = self.stereo_refine_s4(disp_s8, feat_l_s4, max_disp=self.max_disp_s8 * 2.0)
        return disp_s4, disp_s8


# ─────────────────────────────────────────────────────────────────────
# CHANNEL ALIGNER
# ─────────────────────────────────────────────────────────────────────
class ChannelAligner(nn.Module):
    def __init__(self, ch_s4, ch_s8, ch_s16, ch_s32,
                 target_s4=32, target_s8=64, target_s16=128, target_s32=256):
        super().__init__()
        self.align_s4  = nn.Conv2d(ch_s4,  target_s4,  1, bias=False) if ch_s4  != target_s4  else nn.Identity()
        self.align_s8  = nn.Conv2d(ch_s8,  target_s8,  1, bias=False) if ch_s8  != target_s8  else nn.Identity()
        self.align_s16 = nn.Conv2d(ch_s16, target_s16, 1, bias=False) if ch_s16 != target_s16 else nn.Identity()
        self.align_s32 = nn.Conv2d(ch_s32, target_s32, 1, bias=False) if ch_s32 != target_s32 else nn.Identity()
        self.out_channels = (target_s4, target_s8, target_s16, target_s32)
    def forward(self, features):
        return (self.align_s4(features[0]), self.align_s8(features[1]),
                self.align_s16(features[2]), self.align_s32(features[3]))


# ─────────────────────────────────────────────────────────────────────
# NORMALS HEAD
# ─────────────────────────────────────────────────────────────────────
class NormalsHead(nn.Module):
    def __init__(self, ch_s4, ch_s8, deploy=False):
        super().__init__()
        self.deploy    = deploy
        self.s8_adapt  = nn.Conv2d(ch_s8, 32, kernel_size=1)
        fused_ch       = ch_s4 + 32 + 32
        self.stage1    = nn.Sequential(
            CoordConv2d(fused_ch, 96, h=120, w=160, deploy=deploy, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(96, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 32, 3, padding=1), nn.ReLU(inplace=True))
        self.coarse_out = nn.Conv2d(32, 3, 3, padding=1)

    def forward(self, l_s4, l_s8, gray_img, geo_features=None):
        s8_adapted = F.interpolate(self.s8_adapt(l_s8), size=l_s4.shape[2:],
                                   mode='nearest', align_corners=None)
        if geo_features is not None:
            l_s4_combined = torch.cat([l_s4, s8_adapted, geo_features], dim=1)
        else:
            dummy_geo = torch.zeros(l_s4.size(0), 32, l_s4.size(2), l_s4.size(3),
                                    device=l_s4.device, dtype=l_s4.dtype)
            l_s4_combined = torch.cat([l_s4, s8_adapted, dummy_geo], dim=1)
        coarse = self.coarse_out(self.stage1(l_s4_combined))
        if self.deploy:
            v_max, _ = torch.max(torch.abs(coarse), dim=1, keepdim=True)
            scaled   = coarse / (v_max + 1e-6)
            l2       = torch.sqrt(torch.sum(scaled * scaled, dim=1, keepdim=True))
            return coarse / torch.clamp(l2 * (v_max + 1e-6), min=1e-4)
        return F.normalize(coarse, p=2, dim=1, eps=1e-4)


# ─────────────────────────────────────────────────────────────────────
# SEG + YOLO HEADS
# ─────────────────────────────────────────────────────────────────────
class LRASPPHead(nn.Module):
    def __init__(self, low_ch, high_ch, num_classes, normals_ch=3):
        super().__init__()
        self.cbr_high    = nn.Sequential(nn.Conv2d(high_ch, 128, 1, bias=False), nn.BatchNorm2d(128), nn.ReLU(inplace=True))
        self.scale_high  = nn.Sequential(LearnablePool(high_ch, kernel_size=(30,40)),
                                          nn.Conv2d(high_ch, 128, 1, bias=False), nn.Sigmoid())
        self.low_feat_conv  = nn.Conv2d(low_ch,     num_classes, 1)
        self.low_norm_conv  = nn.Conv2d(normals_ch, num_classes, 1)
        self.high_classifier= nn.Conv2d(128,        num_classes, 1)
        self.mid_feat_conv  = nn.Conv2d(128,        num_classes, 3, padding=2, dilation=2)
        self.mid_norm_conv  = nn.Conv2d(normals_ch, num_classes, 3, padding=2, dilation=2)
        self.use_mid = True

    def forward(self, x_low, x_high, normals_s4=None):
        out    = self.cbr_high(x_high) * self.scale_high(x_high)
        out    = F.interpolate(out, scale_factor=4.0, mode='nearest', align_corners=None)
        norm   = normals_s4 if normals_s4 is not None else \
            torch.zeros(x_low.size(0), 3, x_low.size(2), x_low.size(3), device=x_low.device)
        result = self.low_feat_conv(x_low) + self.high_classifier(out) + self.low_norm_conv(norm)
        if self.use_mid:
            norm_mid = normals_s4 if normals_s4 is not None else \
                torch.zeros(out.size(0), 3, out.size(2), out.size(3), device=out.device)
            result = result + self.mid_feat_conv(out) + self.mid_norm_conv(norm_mid)
        return result


class DecoupledHead(nn.Module):
    def __init__(self, ch_in, num_classes, h, w, deploy=False, width=128):
        super().__init__()
        
        self.coord_conv_cls = CoordConv2d(ch_in, width, h=h, w=w, deploy=deploy, kernel_size=3, padding=1)
        self.coord_conv_reg = CoordConv2d(ch_in, width, h=h, w=w, deploy=deploy, kernel_size=3, padding=1)
        
        # 🚨 PATCH V3.0: RepConv integriert die ReLU bereits intern!
        self.cls_convs = nn.Sequential(
            self.coord_conv_cls,
            nn.BatchNorm2d(width),
            nn.ReLU(inplace=True),
            RepConv(width, width) # ⬅️ RepConv statt DWSepConv
        )
        
        self.reg_convs = nn.Sequential(
            self.coord_conv_reg,
            nn.BatchNorm2d(width),
            nn.ReLU(inplace=True),
            RepConv(width, width) # ⬅️ RepConv statt DWSepConv
        )
        
        self.cls_pred = nn.Conv2d(width, num_classes, 1)
        self.reg_pred = nn.Conv2d(width, 4, 1)
        self.obj_pred = nn.Conv2d(width, 1, 1)
    def forward(self, x):
        cls_feat = self.cls_convs(x); reg_feat = self.reg_convs(x)
        return torch.cat([self.reg_pred(reg_feat), self.obj_pred(reg_feat), self.cls_pred(cls_feat)], dim=1)


class YOLOHead(nn.Module):
    def __init__(self, fpn_ch=FPN_CH, num_classes=41, deploy=False):
        super().__init__()
        self.head_s8  = DecoupledHead(fpn_ch, num_classes, h=60, w=80,  deploy=deploy, width=128)
        self.head_s16 = DecoupledHead(fpn_ch, num_classes, h=30, w=40,  deploy=deploy, width=128)
        self.head_s32 = DecoupledHead(fpn_ch, num_classes, h=15, w=20,  deploy=deploy, width=128)
    def forward(self, x_s8, x_s16, x_s32):
        return [self.head_s8(x_s8), self.head_s16(x_s16), self.head_s32(x_s32)]


# ─────────────────────────────────────────────────────────────────────
# FUSED HEXAPOD MODEL V5.0
# ─────────────────────────────────────────────────────────────────────
class FusedHexapodModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.deploy_mode = config.get('deploy', False)
        v5 = config.get('v5', V5_CONFIG)

        self.backbone = timm.create_model('mobilenetv2_140.ra_in1k', pretrained=False,
                                          features_only=True, out_indices=(1,2,3,4))
        ch_s4, ch_s8, ch_s16, ch_s32 = self.backbone.feature_info.channels()

        # 3ch → 1ch grayscale stem
        old_conv = self.backbone.conv_stem
        new_conv = nn.Conv2d(1, old_conv.out_channels, kernel_size=old_conv.kernel_size,
                             stride=old_conv.stride, padding=old_conv.padding, bias=False)
        with torch.no_grad():
            w = old_conv.weight.data
            new_conv.weight.data = w[:,0:1]*0.299 + w[:,1:2]*0.587 + w[:,2:3]*0.114
        self.backbone.conv_stem = new_conv

        # Channel Aligner
        alignment = v5.get('channel_alignment', 'none')
        if alignment == 'moderate':
            self.aligner = ChannelAligner(ch_s4, ch_s8, ch_s16, ch_s32, 32, 64, 128, 256)
            ch_s4, ch_s8, ch_s16, ch_s32 = self.aligner.out_channels
        elif alignment == 'wide':
            self.aligner = ChannelAligner(ch_s4, ch_s8, ch_s16, ch_s32, 64, 128, 256, 512)
            ch_s4, ch_s8, ch_s16, ch_s32 = self.aligner.out_channels
        elif alignment == 'stereo_focused':
            self.aligner = ChannelAligner(ch_s4, ch_s8, ch_s16, ch_s32, 64, 128, 128, 256)
            ch_s4, ch_s8, ch_s16, ch_s32 = self.aligner.out_channels
        else:
            self.aligner = None

        self.sppf = SPPF(c1=ch_s32, c2=ch_s32, k=5)

        disp_steps = config.get('internal_disp_steps', 24)
        if v5.get('use_correlation_stereo', False):
            self.stereo_head = CorrelationStereoHead(ch_s8, ch_s4, max_disp_s8=disp_steps,
                                                      deploy=self.deploy_mode)
        else:
            self.stereo_head = HierarchicalStereoHead(ch_s8, ch_s4, max_disp_s8=disp_steps,
                                                      deploy=self.deploy_mode)

        self.normals_head = NormalsHead(ch_s4, ch_s8, deploy=self.deploy_mode)
        self.seg_head     = LRASPPHead(ch_s4, ch_s16, config['num_seg_classes'], normals_ch=3)
        self.fpn_neck     = LightFPNNeck(ch_s8, ch_s16, ch_s32, fpn_ch=FPN_CH)
        self.geo_stem     = GeometryStem(in_ch=ch_s4, out_ch=32)
        self.yolo_head    = YOLOHead(fpn_ch=FPN_CH, num_classes=config['num_det_classes'],
                                     deploy=self.deploy_mode)

        print(f'  Backbone channels (post-align): s4={ch_s4}, s8={ch_s8}, s16={ch_s16}, s32={ch_s32}')
        print(f'  Total params: {sum(p.numel() for p in self.parameters()):,}')

    def forward(self, x_left, x_right, use_normals_for_stereo=False):
        features_l = self.backbone(x_left)
        if self.aligner: features_l = self.aligner(features_l)

        geo_feat_l  = self.geo_stem(features_l[0])
        normals_s4  = self.normals_head(features_l[0], features_l[1], x_left, geo_features=geo_feat_l)

        if x_right is not None:
            with torch.no_grad():
                features_r = self.backbone(x_right)
                if self.aligner: features_r = self.aligner(features_r)
                geo_feat_r = self.geo_stem(features_r[0])
            normals_for_stereo = normals_s4 if self.deploy_mode else \
                (normals_s4.detach() if use_normals_for_stereo else None)
            disp_s4, disp_s8 = self.stereo_head(
                features_l[1], features_r[1], features_l[0], x_left,
                normals_s4=normals_for_stereo,
                geo_features_l=geo_feat_l, geo_features_r=geo_feat_r)
        else:
            disp_s4, disp_s8 = None, None

        normals_for_others = normals_s4 if self.deploy_mode else \
            (normals_s4.detach() if use_normals_for_stereo else None)
        seg = self.seg_head(features_l[0], features_l[2], normals_s4=normals_for_others)

        f_s32_sppf = self.sppf(features_l[3])
        fpn_s8, fpn_s16, fpn_s32 = self.fpn_neck(features_l[1], features_l[2], f_s32_sppf)
        yolo_s8, yolo_s16, yolo_s32 = self.yolo_head(fpn_s8, fpn_s16, fpn_s32)

        return disp_s4, seg, disp_s8, normals_s4, yolo_s8, yolo_s16, yolo_s32


print('✅ V5.0 Architektur geladen')

✅ V5.0 Architektur geladen


## Zelle 4 — Checkpoint-Konfiguration

**Hier anpassen:** Pfade zu den letzten 5 Checkpoints eintragen.
Die Liste wird nach SWA-Gewichtung verarbeitet (gleichgewichtetes Averaging).

In [35]:
# =====================================================================
# CHECKPOINT-KONFIGURATION — bitte anpassen!
# =====================================================================
SAVE_DIR   = CONFIG['save_dir']   # './checkpoints'
OUT_DIR    = './onnx_split'       # Zielordner für die 3 ONNX-Dateien
DEPLOY_PTH = os.path.join(SAVE_DIR, 'checkpoint_v5_0_deploy.pth')

# ── Option A: Manuell ────────────────────────────────────────────────
# Trage hier exakt die 5 Checkpoints ein, die geittelt werden sollen.
# Die Reihenfolge spielt für SWA keine Rolle (gleiches Gewicht).
CHECKPOINT_PATHS_MANUAL = [
    # Beispiel (durch eigene Pfade ersetzen):
    # f'{SAVE_DIR}/checkpoint_v5_0_best.pth',
    # f'{SAVE_DIR}/checkpoint_v5_0_step_50000.pth',
    # f'{SAVE_DIR}/checkpoint_v5_0_step_48768.pth',
    # f'{SAVE_DIR}/checkpoint_v5_0_step_47536.pth',
    # f'{SAVE_DIR}/checkpoint_v5_0_step_46304.pth',
]

# ── Option B: Automatisch — letzte N step-Checkpoints + optional best ─
N_CHECKPOINTS = 6   # Wie viele Checkpoints sollen geittelt werden?
USE_BEST      = True  # Soll checkpoint_v5_0_best.pth mit einbezogen werden?

def auto_select_checkpoints(save_dir, n=5, use_best=True):
    """Wählt automatisch die n neuesten step-Checkpoints aus."""
    pattern = os.path.join(save_dir, 'checkpoint_v5_0_step_*.pth')
    step_ckpts = sorted(glob.glob(pattern),
                         key=lambda p: int(re.search(r'step_(\d+)', p).group(1)),
                         reverse=True)
    selected = step_ckpts[:n]
    if use_best:
        best = os.path.join(save_dir, 'checkpoint_v5_0_best.pth')
        if os.path.exists(best) and best not in selected:
            selected = [best] + selected[:-1]   # best ersetzt ältesten step-Ckpt
    return selected

# Entscheidung: Manuell oder Auto?
if CHECKPOINT_PATHS_MANUAL:
    CHECKPOINT_PATHS = CHECKPOINT_PATHS_MANUAL
    print('Modus: MANUELL')
else:
    CHECKPOINT_PATHS = auto_select_checkpoints(SAVE_DIR, N_CHECKPOINTS, USE_BEST)
    print('Modus: AUTO')

print(f'\nSWA über {len(CHECKPOINT_PATHS)} Checkpoint(s):')
for p in CHECKPOINT_PATHS:
    exists = '✅' if os.path.exists(p) else '❌ FEHLT'
    print(f'  {exists}  {p}')

if not CHECKPOINT_PATHS:
    print('\n⚠️  Keine Checkpoints gefunden! Bitte CHECKPOINT_PATHS_MANUAL füllen'
          ' oder SAVE_DIR überprüfen.')

Modus: AUTO

SWA über 6 Checkpoint(s):
  ✅  ./checkpoints/checkpoint_v5_0_best.pth
  ✅  ./checkpoints/checkpoint_v5_0_step_105228.pth
  ✅  ./checkpoints/checkpoint_v5_0_step_104720.pth
  ✅  ./checkpoints/checkpoint_v5_0_step_103488.pth
  ✅  ./checkpoints/checkpoint_v5_0_step_102256.pth
  ✅  ./checkpoints/checkpoint_v5_0_step_101024.pth


## Zelle 5 — SWA + RepConv-Folding + Deploy-Mode

In [36]:
# =====================================================================
# HILFSFUNKTIONEN
# =====================================================================

def compute_swa_state_dict(checkpoint_paths):
    """
    Lädt alle Checkpoints und berechnet das gleichgewichtete Mittel
    aller floating-point Parameter. Integer-Tensoren (z.B. running_step)
    werden per Floor-Division gemittelt.
    """
    assert checkpoint_paths, 'Keine Checkpoints übergeben!'
    n = len(checkpoint_paths)
    swa_sd = None

    for i, path in enumerate(checkpoint_paths):
        if not os.path.exists(path):
            raise FileNotFoundError(f'Checkpoint nicht gefunden: {path}')
        print(f'  [{i+1}/{n}] Lade: {os.path.basename(path)}')
        ckpt = torch.load(path, map_location='cpu', weights_only=False)
        # Unterstütze unterschiedliche Checkpoint-Formate
        sd = ckpt.get('model_state_dict', ckpt.get('model_state', ckpt))

        if swa_sd is None:
            swa_sd = {k: v.clone().float() if v.is_floating_point() else v.clone()
                      for k, v in sd.items()}
        else:
            for k in swa_sd:
                if k in sd:
                    if swa_sd[k].is_floating_point():
                        swa_sd[k] += sd[k].float()
                    else:
                        swa_sd[k] += sd[k]

    # Durchschnitt bilden
    for k in swa_sd:
        if swa_sd[k].is_floating_point():
            swa_sd[k] /= n
        else:
            swa_sd[k] = torch.div(swa_sd[k], n, rounding_mode='floor')

    return swa_sd

'''
def fold_repconv(model):
    """Ruft switch_to_deploy() auf allen RepConv-Layern auf."""
    count = 0
    for name, m in model.named_modules():
        if isinstance(m, RepConv) and not m.deploy:
            m.switch_to_deploy()
            count += 1
    return count

'''
def fold_repconv(model):
    # RepConvs in Detection-relevanten Modulen NICHT falten
    detection_modules = {'fpn_neck', 'yolo_head'}
    fold_count = 0
    for name, m in model.named_modules():
        if isinstance(m, RepConv) and not m.deploy:
            # Nur GeoStem falten, FPN/YOLO nicht
            if any(dm in name for dm in detection_modules):
                print(f"  ✅ Skipped: {name}")
                continue  # Skip!
            m.switch_to_deploy()
            fold_count += 1
    return fold_count

'''
def enable_deploy_flags(model):
    """Setzt alle .deploy- und .deploy_mode-Flags auf True."""
    if hasattr(model, 'deploy_mode'):
        model.deploy_mode = True
    for m in model.modules():
        if hasattr(m, 'deploy'):
            m.deploy = True
'''
def enable_deploy_flags(model):
    """Setzt alle .deploy- und .deploy_mode-Flags auf True (schützt aber RepConvs)."""
    if hasattr(model, 'deploy_mode'):
        model.deploy_mode = True
        
    for m in model.modules():
        # deploy_mode setzen (z.B. für SPPF oder CoordConv)
        if hasattr(m, 'deploy_mode'):
            m.deploy_mode = True
            
        # deploy setzen, ABER RepConv ignorieren!
        if hasattr(m, 'deploy'):
            # RepConv verwaltet sein Flag selbst. Wir fassen es hier nicht an!
            if type(m).__name__ != 'RepConv':
                m.deploy = True
            else:
                print(f"  ✅ Deploy-flag skipped for {m}")

def load_state_smart(model, state_dict):
    """
    Lädt state_dict mit shape-gefiltertem strict=False.
    Überspringt Keys, die fehlen oder eine andere Shape haben.
    CoordConv-Koordinatenbuffer ('y_coords', 'x_coords') werden
    bewusst ausgelassen (werden beim Modell-Init neu berechnet).
    """
    model_sd = model.state_dict()
    filtered = {}
    skipped_shape, skipped_missing = [], []

    for k, v in state_dict.items():
        if k not in model_sd:
            skipped_missing.append(k)
            continue
        if v.shape != model_sd[k].shape:
            skipped_shape.append(f'{k}: {v.shape} vs {model_sd[k].shape}')
            continue
        filtered[k] = v

    missing_after, unexpected = model.load_state_dict(filtered, strict=False)
    # Koordinatenbuffer sind nicht im state_dict → ignorieren
    real_missing = [k for k in missing_after
                    if 'y_coords' not in k and 'x_coords' not in k
                    and 'disp_reg' not in k]

    print(f'  Geladene Keys:         {len(filtered)}')
    if skipped_shape:   print(f'  ⚠️  Shape-Mismatch:    {len(skipped_shape)}  →  {skipped_shape[:3]}')
    if skipped_missing: print(f'  ⚠️  Nicht im Modell:   {len(skipped_missing)} →  {skipped_missing[:3]}')
    if real_missing:    print(f'  ⚠️  Fehlende Keys:     {len(real_missing)}    →  {real_missing[:3]}')


# =====================================================================
# AUSFÜHRUNG: SWA → FOLDING → DEPLOY-FLAG → CHECKPOINT SPEICHERN
# =====================================================================
print('='*60)
print(f'  SWA über {len(CHECKPOINT_PATHS)} Checkpoint(s)')
print('='*60)

# 1. Modell in TRAININGS-Struktur erstellen (deploy=False!)
#    Nur so haben RepConv-Layer ihre 3 Branches für das Folding.
train_config = copy.deepcopy(CONFIG)
train_config['deploy'] = False

print('\n① Erstelle Modell (Trainings-Struktur, deploy=False)...')
model = FusedHexapodModel(train_config)

# 2. SWA: Gemitteltes state_dict laden
print(f'\n② SWA — {len(CHECKPOINT_PATHS)} Checkpoint(s) werden gemittelt...')
swa_state_dict = compute_swa_state_dict(CHECKPOINT_PATHS)
print('   Lade SWA-Gewichte ins Modell...')
load_state_smart(model, swa_state_dict)

# 3. Eval-Modus setzen (BatchNorm-Statistics einfrieren)
model.eval()

# 4. RepConv-Folding
print('\n③ RepConv-Folding (switch_to_deploy)...')
n_folded = fold_repconv(model)
print(f'   ✅ {n_folded} RepConv-Layer gefaltet')

# 5. Alle deploy-Flags auf True setzen
print('\n④ Deploy-Flags aktivieren...')
enable_deploy_flags(model)
print('   ✅ Alle .deploy- und .deploy_mode-Flags gesetzt')

# 6. Deploy-Checkpoint speichern
os.makedirs(SAVE_DIR, exist_ok=True)
torch.save(model.state_dict(), DEPLOY_PTH)
size_mb = os.path.getsize(DEPLOY_PTH) / 1e6
print(f'\n⑤ Deploy-Checkpoint gespeichert: {DEPLOY_PTH} ({size_mb:.1f} MB)')

print('\n✅ SWA + Folding + Deploy abgeschlossen.')

  SWA über 6 Checkpoint(s)

① Erstelle Modell (Trainings-Struktur, deploy=False)...
  Backbone channels (post-align): s4=64, s8=128, s16=128, s32=256
  Total params: 6,005,888

② SWA — 6 Checkpoint(s) werden gemittelt...
  [1/6] Lade: checkpoint_v5_0_best.pth
  [2/6] Lade: checkpoint_v5_0_step_105228.pth
  [3/6] Lade: checkpoint_v5_0_step_104720.pth
  [4/6] Lade: checkpoint_v5_0_step_103488.pth
  [5/6] Lade: checkpoint_v5_0_step_102256.pth
  [6/6] Lade: checkpoint_v5_0_step_101024.pth
   Lade SWA-Gewichte ins Modell...
  Geladene Keys:         649

③ RepConv-Folding (switch_to_deploy)...
  ✅ Skipped: fpn_neck.smooth_s8
  ✅ Skipped: fpn_neck.smooth_s16
  ✅ Skipped: fpn_neck.bu_s16
  ✅ Skipped: fpn_neck.bu_s32
  ✅ Skipped: yolo_head.head_s8.cls_convs.3
  ✅ Skipped: yolo_head.head_s8.reg_convs.3
  ✅ Skipped: yolo_head.head_s16.cls_convs.3
  ✅ Skipped: yolo_head.head_s16.reg_convs.3
  ✅ Skipped: yolo_head.head_s32.cls_convs.3
  ✅ Skipped: yolo_head.head_s32.reg_convs.3
   ✅ 2 RepConv-Layer

## Zelle 6 — Kanal-Dimensionen auslesen

In [37]:
# Backbone-Ausgabe-Shapes nach dem Aligner ermitteln
# (nötig für korrekte Dummy-Inputs beim ONNX-Export)
model_cpu = model.cpu().eval()

with torch.no_grad():
    _feats = model_cpu.backbone(torch.zeros(1, 1, 480, 640))
    if model_cpu.aligner:
        _feats = model_cpu.aligner(_feats)

CH_S4, CH_S8, CH_S16, CH_S32 = [f.shape[1] for f in _feats]
del _feats

print(f'Kanal-Dimensionen (nach Alignment):')
print(f'  s4  → {CH_S4}  ch   @ [1, {CH_S4}, 120, 160]')
print(f'  s8  → {CH_S8}  ch   @ [1, {CH_S8},  60,  80]')
print(f'  s16 → {CH_S16}  ch  @ [1, {CH_S16},  30,  40]')
print(f'  s32 → {CH_S32}  ch  @ [1, {CH_S32},  15,  20]')

Kanal-Dimensionen (nach Alignment):
  s4  → 64  ch   @ [1, 64, 120, 160]
  s8  → 128  ch   @ [1, 128,  60,  80]
  s16 → 128  ch  @ [1, 128,  30,  40]
  s32 → 256  ch  @ [1, 256,  15,  20]


## Zelle 7 — 3-HEF ONNX-Export

| HEF | Modul | Inputs | Outputs |
|-----|-------|--------|---------|
| **Backbone** | Backbone + ChannelAligner | `gray_img [1,1,480,640]` | `f_s4, f_s8, f_s16, f_s32` |
| **Geometry** | GeoStem + NormalsHead + CorrelationStereoHead | `f_s4_l, f_s8_l, f_s4_r, f_s8_r` | `disp_s4, normals_s4, disp_s8` |
| **Detection** | SPPF + FPN + LRASPPHead + YOLOHead | `f_s4_l, f_s8_l, f_s16_l, f_s32_l, normals_s4` | `seg, yolo_s8, yolo_s16, yolo_s32` |

In [41]:
import subprocess
os.makedirs(OUT_DIR, exist_ok=True)


# ─────────────────────────────────────────────────────────────────────
# WRAPPER-KLASSEN
# ─────────────────────────────────────────────────────────────────────

class BackboneWrapper(nn.Module):
    """
    HEF A — Backbone
    Input:  gray_img [1, 1, 480, 640]
    Output: f_s4, f_s8, f_s16, f_s32
    """
    def __init__(self, m):
        super().__init__()
        self.backbone = m.backbone
        self.aligner  = m.aligner   # None wenn kein Alignment

    def forward(self, gray_img):
        f = self.backbone(gray_img)
        if self.aligner:
            f = self.aligner(f)
        return f[0], f[1], f[2], f[3]


class GeometryWrapper(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.geo_stem = m.geo_stem
        self.normals_head = m.normals_head
        self.stereo_head = m.stereo_head
        
    
    def forward(self, f_s4_l, f_s8_l, f_s4_r, f_s8_r):
        geo_feat_l = self.geo_stem(f_s4_l)
        geo_feat_r = self.geo_stem(f_s4_r)
        normals_s4 = self.normals_head(f_s4_l, f_s8_l, None, geo_features=geo_feat_l)
        disp_s4, disp_s8 = self.stereo_head(
            f_s8_l, f_s8_r, f_s4_l, None,
            normals_s4=normals_s4,
            geo_features_l=geo_feat_l, geo_features_r=geo_feat_r)
        return disp_s4, normals_s4, disp_s8


class DetectionWrapper(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.sppf = m.sppf
        self.fpn_neck = m.fpn_neck
        self.seg_head = m.seg_head
        self.yolo_head = m.yolo_head
            
    def forward(self, f_s4_l, f_s8_l, f_s16_l, f_s32_l, normals_s4):
        seg = self.seg_head(f_s4_l, f_s16_l, normals_s4=normals_s4)
        f_s32_sppf = self.sppf(f_s32_l)
        fpn_s8, fpn_s16, fpn_s32 = self.fpn_neck(f_s8_l, f_s16_l, f_s32_sppf)
        yolo_s8, yolo_s16, yolo_s32 = self.yolo_head(fpn_s8, fpn_s16, fpn_s32)
        return seg, yolo_s8, yolo_s16, yolo_s32


# ─────────────────────────────────────────────────────────────────────
# EXPORT-HILFSFUNKTION
# ─────────────────────────────────────────────────────────────────────

def export_onnx_split(module, dummy_inputs, input_names, output_names, basename):
    """
    Exportiert module nach ONNX (opset 13) und vereinfacht mit onnxsim.
    Gibt den Pfad zur finalen Datei zurück (simplified wenn verfügbar).
    """
    raw_path = os.path.join(OUT_DIR, f'{basename}.onnx')
    sim_path = os.path.join(OUT_DIR, f'{basename}_simplified.onnx')

    print(f'\n  ▶ {basename}.onnx')
    module.eval()
    with torch.no_grad():
        torch.onnx.export(
            module,
            dummy_inputs,
            raw_path,
            input_names=input_names,
            output_names=output_names,
            opset_version=13,
            do_constant_folding=True,
        )
    raw_mb = os.path.getsize(raw_path) / 1e6
    print(f'    Raw:        {raw_mb:.1f} MB')

    try:
        result = subprocess.run(
            ['python', '-m', 'onnxsim', raw_path, sim_path],
            check=True, capture_output=True, text=True, timeout=300)
        sim_mb = os.path.getsize(sim_path) / 1e6
        print(f'    Simplified: {sim_mb:.1f} MB  ✅')
        return sim_path
    except subprocess.CalledProcessError as e:
        print(f'    ⚠️  onnxsim fehlgeschlagen: {e.stderr.strip()[:200]}')
        print(f'    Verwende Raw-ONNX')
        return raw_path
    except subprocess.TimeoutExpired:
        print(f'    ⚠️  onnxsim Timeout — verwende Raw-ONNX')
        return raw_path


# ─────────────────────────────────────────────────────────────────────
# EXPORT ALLER 3 HEFs
# ─────────────────────────────────────────────────────────────────────

print('='*60)
print('  3-HEF ONNX SPLIT EXPORT')
print('='*60)

model_cpu = model.cpu().eval()

# ──── HEF A: Backbone ────────────────────────────────────────────────
bb_wrapper = BackboneWrapper(model_cpu).eval()
bb_path = export_onnx_split(
    bb_wrapper,
    dummy_inputs=(torch.randn(1, 1, 480, 640),),
    input_names=['input_layer1'],
    output_names=['f_s4', 'f_s8', 'f_s16', 'f_s32'],
    basename='hexapod_v5_backbone')
del bb_wrapper

# ──── HEF B: Geometry ────────────────────────────────────────────────
geo_wrapper = GeometryWrapper(model_cpu).eval()
geo_path = export_onnx_split(
    geo_wrapper,
    dummy_inputs=(
        torch.randn(1, CH_S4, 120, 160),   # f_s4_l
        torch.randn(1, CH_S8,  60,  80),   # f_s8_l
        torch.randn(1, CH_S4, 120, 160),   # f_s4_r
        torch.randn(1, CH_S8,  60,  80),   # f_s8_r
    ),
    input_names=['f_s4_l', 'f_s8_l', 'f_s4_r', 'f_s8_r'],
    output_names=['disp_s4', 'normals_s4', 'disp_s8'],
    basename='hexapod_v5_geometry')
del geo_wrapper

# ──── HEF C: Detection ───────────────────────────────────────────────
det_wrapper = DetectionWrapper(model_cpu).eval()
det_path = export_onnx_split(
    det_wrapper,
    dummy_inputs=(
        torch.randn(1, CH_S4,  120, 160),  # f_s4_l
        torch.randn(1, CH_S8,   60,  80),  # f_s8_l
        torch.randn(1, CH_S16,  30,  40),  # f_s16_l
        torch.randn(1, CH_S32,  15,  20),  # f_s32_l
        torch.randn(1, 3,      120, 160),  # normals_s4
    ),
    input_names=['f_s4_l', 'f_s8_l', 'f_s16_l', 'f_s32_l', 'normals_s4'],
    output_names=['seg', 'yolo_s8', 'yolo_s16', 'yolo_s32'],
    basename='hexapod_v5_detection')
del det_wrapper


# ─────────────────────────────────────────────────────────────────────
# ZUSAMMENFASSUNG
# ─────────────────────────────────────────────────────────────────────
print('\n' + '='*60)
print('  EXPORT ABGESCHLOSSEN')
print('='*60)

hef_files = [
    ('Backbone',  'hexapod_v5_backbone_simplified.onnx',  'hexapod_v5_backbone.onnx'),
    ('Geometry',  'hexapod_v5_geometry_simplified.onnx',  'hexapod_v5_geometry.onnx'),
    ('Detection', 'hexapod_v5_detection_simplified.onnx', 'hexapod_v5_detection.onnx'),
]

total_mb = 0
for label, sim_name, raw_name in hef_files:
    sim_p = os.path.join(OUT_DIR, sim_name)
    raw_p = os.path.join(OUT_DIR, raw_name)
    path  = sim_p if os.path.exists(sim_p) else raw_p
    if os.path.exists(path):
        mb = os.path.getsize(path) / 1e6
        total_mb += mb
        tag = '(simplified)' if '_simplified' in path else '(raw)'
        print(f'  {label:10s}  {mb:6.1f} MB  {tag}  →  {os.path.basename(path)}')
    else:
        print(f'  {label:10s}  ❌ Datei fehlt')

print(f'  {"─"*50}')
print(f'  Gesamt:    {total_mb:6.1f} MB')
print(f'\nNächster Schritt: Hailo DFC Quantisierung')
print(f'  hailo quantize hexapod_v5_backbone_simplified.onnx ...')
print(f'  hailo quantize hexapod_v5_geometry_simplified.onnx ...')
print(f'  hailo quantize hexapod_v5_detection_simplified.onnx ...')

  3-HEF ONNX SPLIT EXPORT

  ▶ hexapod_v5_backbone.onnx
    Raw:        14.5 MB
    Simplified: 14.5 MB  ✅

  ▶ hexapod_v5_geometry.onnx


/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/torch/onnx/_internal/jit_utils.py:308: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at ../torch/csrc/jit/passes/onnx/constant_fold.cpp:178.)
  _C._jit_pass_onnx_node_shape_type_inference(node, params_dict, opset_version)
/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/torch/onnx/utils.py:663: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at ../torch/csrc/jit/passes/onnx/constant_fold.cpp:178.)
  _C._jit_pass_onnx_graph_shape_type_inference(
/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/torch/onnx/utils.py:1186: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at ../to

    Raw:        1.6 MB
    Simplified: 1.6 MB  ✅

  ▶ hexapod_v5_detection.onnx
    Raw:        8.2 MB
    Simplified: 8.2 MB  ✅

  EXPORT ABGESCHLOSSEN
  Backbone      14.5 MB  (simplified)  →  hexapod_v5_backbone_simplified.onnx
  Geometry       1.6 MB  (simplified)  →  hexapod_v5_geometry_simplified.onnx
  Detection      8.2 MB  (simplified)  →  hexapod_v5_detection_simplified.onnx
  ──────────────────────────────────────────────────
  Gesamt:      24.3 MB

Nächster Schritt: Hailo DFC Quantisierung
  hailo quantize hexapod_v5_backbone_simplified.onnx ...
  hailo quantize hexapod_v5_geometry_simplified.onnx ...
  hailo quantize hexapod_v5_detection_simplified.onnx ...


## Zelle 8 — Numerische Validierung (PyTorch vs ONNX Runtime)

Prüft ob alle 3 ONNX-Modelle numerisch korrekte Outputs liefern.

In [43]:
try:
    import onnxruntime as ort
    import onnx
    ORT_AVAILABLE = True
except ImportError:
    print('⚠️  onnxruntime nicht installiert (pip install onnxruntime). Validation übersprungen.')
    ORT_AVAILABLE = False

if ORT_AVAILABLE:
    torch.manual_seed(0)
    np.random.seed(0)

    dummy_img   = torch.randn(1, 1, 480, 640)
    dummy_f_s4l = torch.randn(1, CH_S4, 120, 160)
    dummy_f_s8l = torch.randn(1, CH_S8,  60,  80)
    dummy_f_s4r = torch.randn(1, CH_S4, 120, 160)
    dummy_f_s8r = torch.randn(1, CH_S8,  60,  80)
    dummy_f_s16 = torch.randn(1, CH_S16, 30,  40)
    dummy_f_s32 = torch.randn(1, CH_S32, 15,  20)
    dummy_norms = torch.randn(1, 3,     120, 160)

    model_cpu = model.cpu().eval()

    def check_hef(onnx_path, pt_fn, ort_inputs, label, tol=1e-4):
        if not os.path.exists(onnx_path):
            print(f'  {label}: ❌ Datei nicht gefunden: {onnx_path}')
            return

        # ONNX Validity
        try:
            onnx.checker.check_model(onnx.load(onnx_path))
        except Exception as e:
            print(f'  {label}: ❌ ONNX Graph-Fehler: {e}')
            return

        # ORT Inferenz
        sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
        ort_out = sess.run(None, ort_inputs)

        # PyTorch Inferenz
        with torch.no_grad():
            pt_out = pt_fn()
        if not isinstance(pt_out, (list, tuple)):
            pt_out = [pt_out]

        print(f'  {label}:')
        all_ok = True
        for i, (pt, ot) in enumerate(zip(pt_out, ort_out)):
            if pt is None: continue
            pt_np   = pt.numpy() if isinstance(pt, torch.Tensor) else pt
            max_diff = np.abs(pt_np - ot).max()
            ok       = max_diff < tol
            all_ok   = all_ok and ok
            status   = '✅' if ok else ('⚠️' if max_diff < 1e-2 else '❌')
            out_name = sess.get_outputs()[i].name
            print(f'    {status}  {out_name:15s}: max_diff = {max_diff:.2e}')

        if all_ok:
            print(f'    → Alle Outputs numerisch identisch (tol={tol})')

    print('='*60)
    print('  NUMERISCHE VALIDIERUNG')
    print('='*60)

    # ── HEF A: Backbone ──────────────────────────────────────────────
    bb_onnx = os.path.join(OUT_DIR, 'hexapod_v5_backbone_simplified.onnx')
    if not os.path.exists(bb_onnx):
        bb_onnx = os.path.join(OUT_DIR, 'hexapod_v5_backbone.onnx')

    _bb_mod = BackboneWrapper(model_cpu).eval()
    check_hef(
        bb_onnx,
        pt_fn=lambda: _bb_mod(dummy_img),
        ort_inputs={'input_layer1': dummy_img.numpy()},
        label='Backbone')

    # ── HEF B: Geometry ──────────────────────────────────────────────
    geo_onnx = os.path.join(OUT_DIR, 'hexapod_v5_geometry_simplified.onnx')
    if not os.path.exists(geo_onnx):
        geo_onnx = os.path.join(OUT_DIR, 'hexapod_v5_geometry.onnx')

    _geo_mod = GeometryWrapper(model_cpu).eval()
    check_hef(
        geo_onnx,
        pt_fn=lambda: _geo_mod(dummy_f_s4l, dummy_f_s8l, dummy_f_s4r, dummy_f_s8r),
        ort_inputs={'f_s4_l': dummy_f_s4l.numpy(), 'f_s8_l': dummy_f_s8l.numpy(),
                    'f_s4_r': dummy_f_s4r.numpy(), 'f_s8_r': dummy_f_s8r.numpy()},
        label='Geometry')

    # ── HEF C: Detection ─────────────────────────────────────────────
    det_onnx = os.path.join(OUT_DIR, 'hexapod_v5_detection_simplified.onnx')
    if not os.path.exists(det_onnx):
        det_onnx = os.path.join(OUT_DIR, 'hexapod_v5_detection.onnx')

    _det_mod = DetectionWrapper(model_cpu).eval()
    check_hef(
        det_onnx,
        pt_fn=lambda: _det_mod(dummy_f_s4l, dummy_f_s8l, dummy_f_s16, dummy_f_s32, dummy_norms),
        ort_inputs={'f_s4_l':   dummy_f_s4l.numpy(), 'f_s8_l': dummy_f_s8l.numpy(),
                    'f_s16_l':  dummy_f_s16.numpy(),  'f_s32_l': dummy_f_s32.numpy(),
                    'normals_s4': dummy_norms.numpy()},
        label='Detection')

    print('\n✅ Validierung abgeschlossen')

  NUMERISCHE VALIDIERUNG
  Backbone:
    ✅  f_s4           : max_diff = 4.35e-05
    ✅  f_s8           : max_diff = 3.42e-05
    ✅  f_s16          : max_diff = 2.74e-05
    ✅  f_s32          : max_diff = 4.39e-05
    → Alle Outputs numerisch identisch (tol=0.0001)
  Geometry:
    ✅  disp_s4        : max_diff = 5.15e-05
    ✅  normals_s4     : max_diff = 2.41e-06
    ✅  disp_s8        : max_diff = 3.53e-05
    → Alle Outputs numerisch identisch (tol=0.0001)
  Detection:
    ✅  seg            : max_diff = 1.67e-05
    ✅  yolo_s8        : max_diff = 1.81e-05
    ✅  yolo_s16       : max_diff = 1.72e-05
    ✅  yolo_s32       : max_diff = 1.34e-05
    → Alle Outputs numerisch identisch (tol=0.0001)

✅ Validierung abgeschlossen


## Zelle 9 — ONNX Graph-Inspektion (optional)

Zeigt Inputs/Outputs und Opset-Version aller 3 Dateien.

In [44]:
try:
    import onnx

    for label, basename in [('Backbone',  'hexapod_v5_backbone'),
                              ('Geometry',  'hexapod_v5_geometry'),
                              ('Detection', 'hexapod_v5_detection')]:
        sim_p = os.path.join(OUT_DIR, f'{basename}_simplified.onnx')
        raw_p = os.path.join(OUT_DIR, f'{basename}.onnx')
        path  = sim_p if os.path.exists(sim_p) else raw_p
        if not os.path.exists(path):
            print(f'{label}: ❌ nicht gefunden')
            continue

        m = onnx.load(path)
        opset = m.opset_import[0].version
        print(f'\n── {label}  (opset {opset},  {os.path.getsize(path)/1e6:.1f} MB) ──')
        print('  Inputs:')
        for inp in m.graph.input:
            shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
            print(f'    {inp.name:20s}  {shape}')
        print('  Outputs:')
        for out in m.graph.output:
            shape = [d.dim_value for d in out.type.tensor_type.shape.dim]
            print(f'    {out.name:20s}  {shape}')

except ImportError:
    print('onnx nicht installiert — Inspektion übersprungen')


── Backbone  (opset 13,  14.5 MB) ──
  Inputs:
    input_layer1          [1, 1, 480, 640]
  Outputs:
    f_s4                  [1, 64, 120, 160]
    f_s8                  [1, 128, 60, 80]
    f_s16                 [1, 128, 30, 40]
    f_s32                 [1, 256, 15, 20]

── Geometry  (opset 13,  1.6 MB) ──
  Inputs:
    f_s4_l                [1, 64, 120, 160]
    f_s8_l                [1, 128, 60, 80]
    f_s4_r                [1, 64, 120, 160]
    f_s8_r                [1, 128, 60, 80]
  Outputs:
    disp_s4               [1, 1, 120, 160]
    normals_s4            [1, 3, 120, 160]
    disp_s8               [1, 1, 60, 80]

── Detection  (opset 13,  8.2 MB) ──
  Inputs:
    f_s4_l                [1, 64, 120, 160]
    f_s8_l                [1, 128, 60, 80]
    f_s16_l               [1, 128, 30, 40]
    f_s32_l               [1, 256, 15, 20]
    normals_s4            [1, 3, 120, 160]
  Outputs:
    seg                   [1, 6, 120, 160]
    yolo_s8               [1, 46, 60, 80]
    yo